# Ensemble mean

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/L16B06")

## Load training and test datasets

In [ ]:
import xarray as xr

## Open a netCDF file in a xarray dataset
fname = 'data/garachico256.ens.nc'
ds    = xr.open_dataset(fname)
train = ds['tephra_col_mass']

fname = 'data/garachico2048.ens.nc'
ds    = xr.open_dataset(fname)
test  = ds['tephra_col_mass']

## Load a pre-trained VAE

In [ ]:
import torch
from modules.model import VariationalAutoencoder
from modules.dataset import MinMaxScale

## Load weight parameters and some metadata
fname = output_dir / 'model.pt'
checkpoint = torch.load(fname)

## Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

## Generate a VAE ensemble

In [ ]:
## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

## Generate nens new samples
nens = 5000
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()

## Plot configuration

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as crs                        # coordinate systems for maps
import cartopy.feature as cfeature

plt.rcParams['figure.dpi'] = 200

BORDERS = cfeature.NaturalEarthFeature(
        scale     = '10m',
        category  = 'cultural',
        name      = 'admin_0_countries',
        edgecolor = 'gray',
        facecolor = 'none'
        )
LAND = cfeature.NaturalEarthFeature(
        'physical', 'land', '10m',
        edgecolor = 'none',
        facecolor = 'lightgrey',
        alpha     = 0.8
        )

conf1 = dict(cmap='hot', levels=[5,10,15,20,25,30,35,40], extend='max')
conf2 = dict(cmap='bwr', vmin=-2.0, vmax=2.0)

## Plot ensemble mean

In [ ]:
## Data
x_train = train.mean(dim='ens')
x_vae   = x.mean(0)
x_test  = test.mean(dim='ens')

lat = ds.lat
lon = ds.lon

In [ ]:
fig, axs = plt.subplots(ncols=4,
                        sharex=True,
                        sharey=True,
                        subplot_kw={'projection': crs.PlateCarree()}, 
                        figsize=(14,7),
                       )

_  = axs[0].contourf(lon,lat,x_train,**conf1)
cs = axs[1].contourf(lon,lat,x_vae,**conf1)

cbar = fig.colorbar(cs,
                    ax=axs[:2],
                    shrink = 0.5,
                    pad = 0.05,
                    orientation = 'horizontal',
                    label = r'Ensemble mean [$g~m^{-2}$]')

_  = axs[2].pcolormesh(lon,lat,x_train-x_test,**conf2)
cs = axs[3].pcolormesh(lon,lat,x_vae-x_test,**conf2)

cbar = fig.colorbar(cs,
                    ax=axs[2:],
                    shrink = 0.5,
                    pad = 0.05,
                    orientation = 'horizontal',
                    label = r'Bias [$g~m^{-2}$]')

titles = ['Training dataset','VAE','Training dataset','VAE']
labels = (c for c in 'abcd')
for i, ax in enumerate(axs.flat):
    ax.set_title(f'({next(labels)}) {titles[i]}')
    ax.set_extent([-20, -11, 23, 30]) # [x1,x2,y1,y2]
    ax.add_feature(LAND,zorder=0)
    ax.add_feature(BORDERS, linewidth=0.4)
    ###
    ### Add grid lines
    ###
    gl = ax.gridlines(
        crs         = crs.PlateCarree(),
        draw_labels = ['left','bottom'],
        linewidth   = 0.5,
        color       = 'gray',
        alpha       = 0.5,
        linestyle   = '--')
    gl.xlabel_style  = {'size': 8}
    gl.ylabel_style  = {'rotation': 89, 'size': 8}